# Hybrid Ranking Model

This notebook trains the hybrid ranking model.

It uses the features created in the content feature notebook and learns how to score user–movie pairs.

The model learns from:

- movie genre features

- user genre profiles

- user-movie similarity

- ALS scores

The goal is to rank movies so that positive user interactions appear higher than negatives.

A gradient boosting model is used because it handles mixed features well and is simple to train.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRanker
import lightgbm as lgb
from sklearn.metrics import ndcg_score

# Load the hybrid training data

The file created in the previous notebook contains labeled user–movie pairs.

Each row has:

- user index

- movie index

- label (1 or 0)

- genre features

- taste profile features

- similarity score

- ALS score

This notebook loads that file and prepares it for the ranking model.

In [2]:
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [3]:
pairs = pd.read_parquet(PROCESSED_DIR / "hybrid_train_pairs.parquet")

In [4]:
pairs.head()

,u_index,m_index,label,genre_Action_user,genre_Adventure_user,genre_Animation_user,genre_Children's_user,genre_Comedy_user,genre_Crime_user,genre_Documentary_user,...,genre_Horror_movie,genre_Musical_movie,genre_Mystery_movie,genre_Romance_movie,genre_Sci-Fi_movie,genre_Thriller_movie,genre_War_movie,genre_Western_movie,genre_sim,als_score
0,0,31,1,0.116279,0.093023,0.186047,0.255814,0.209302,0.046512,0.0,...,0,0,0,0,0,0,0,0,0.698836,0.216864
1,0,22,1,0.116279,0.093023,0.186047,0.255814,0.209302,0.046512,0.0,...,0,0,0,0,1,0,0,0,0.282372,0.700850
2,0,27,1,0.116279,0.093023,0.186047,0.255814,0.209302,0.046512,0.0,...,0,0,0,1,0,0,0,0,0.611807,0.764807
3,0,37,1,0.116279,0.093023,0.186047,0.255814,0.209302,0.046512,0.0,...,0,1,0,0,0,0,0,0,0.557177,-0.398312
4,0,24,1,0.116279,0.093023,0.186047,0.255814,0.209302,0.046512,0.0,...,0,0,0,1,0,0,0,0,0.166390,-0.122625


# Prepare data for LightGBM ranking

The LightGBM ranking model needs group information.

A group is a set of items that belong to the same user.

Steps:

- Sort rows by user

- Create a list with the number of rows per user

- Split into train and validation sets

The label column is used to teach the model which movies should be ranked higher.

In [5]:
pairs = pairs.sort_values("u_index").reset_index(drop=True)

In [6]:
group_sizes = pairs.groupby("u_index").size().values

In [7]:
print("Number of users:", len(group_sizes))
print("Total rows:", len(pairs))

Number of users: 6040
Total rows: 1400513


# Selecting feature columns

The model does not use the raw id columns.

It uses the numeric feature columns only.

In [8]:
feature_cols = [c for c in pairs.columns if c not in ["label", "u_index", "m_index"]]
print("Number of features:", len(feature_cols))

Number of features: 38


In [9]:
feature_cols[:10]

['genre_Action_user',
 'genre_Adventure_user',
 'genre_Animation_user',
 "genre_Children's_user",
 'genre_Comedy_user',
 'genre_Crime_user',
 'genre_Documentary_user',
 'genre_Drama_user',
 'genre_Fantasy_user',
 'genre_Film-Noir_user']

# Preparing LightGBM datasets

The data is split into train and validation sets.

Validation helps monitor model performance and avoids overfitting.

Group information must also be split in the same order.

In [10]:
# Extract inputs and target
X = pairs[feature_cols].values
y = pairs["label"].values

In [11]:
# Train-test split on row level
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

In [12]:
# Group by user in each split
train_df = pairs.iloc[X_train[:,0].argsort()] if False else pairs

The ranking model needs group information.

Since the data was shuffled, group information must be recomputed.

A simple fix is to avoid shuffling by user and split by user index.

This keeps group structure correct.

In [13]:
# Split by user instead of row
users = pairs["u_index"].unique()
train_users, val_users = train_test_split(users, test_size=0.2, random_state=42)

In [14]:
train_data = pairs[pairs["u_index"].isin(train_users)]
val_data = pairs[pairs["u_index"].isin(val_users)]

In [15]:
X_train = train_data[feature_cols].values
y_train = train_data["label"].values
X_val = val_data[feature_cols].values
y_val = val_data["label"].values

In [16]:
train_group = train_data.groupby("u_index").size().values
val_group = val_data.groupby("u_index").size().values

In [17]:
print("Train rows:", len(train_data))
print("Val rows:", len(val_data))
print("Train groups:", len(train_group))
print("Val groups:", len(val_group))

Train rows: 1126848
Val rows: 273665
Train groups: 4832
Val groups: 1208


# Training the LightGBM ranker

The model is set up with simple parameters.

LightGBM will learn to rank movies higher if the user interacted with them.

In [18]:
ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

In [19]:
ranker.fit(
    X_train, y_train,
    group=train_group,
    eval_set=[(X_val, y_val)],
    eval_group=[val_group],
    eval_at=[5, 10]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.081446 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5136
[LightGBM] [Info] Number of data points in the train set: 1126848, number of used features: 38


LGBMRanker(learning_rate=0.05, metric='ndcg', n_estimators=200,
           objective='lambdarank', random_state=42)

# Evaluating the model

Evaluation is done using NDCG.

NDCG measures how well the ranking places positive items near the top.

In [20]:
# Predict scores for validation set
y_val_pred = ranker.predict(X_val)

/Users/sanjaydilip/Desktop/Code/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


In [21]:
# Compute NDCG@10
val_data_sorted = val_data.copy()
val_data_sorted["pred"] = y_val_pred
val_data_sorted = val_data_sorted.sort_values(["u_index", "pred"], ascending=[True, False])

In [22]:
ndcgs = []

In [23]:
for user, group in val_data_sorted.groupby("u_index"):
    y_true = group["label"].values.reshape(1, -1)
    y_pred = group["pred"].values.reshape(1, -1)
    score = ndcg_score(y_true, y_pred, k=10)
    ndcgs.append(score)

In [24]:
print("Average NDCG:", np.mean(ndcgs))

Average NDCG: 0.8770847561443837


# Saving the hybrid model

The trained model is stored for use in the final app.

In [25]:
model_path = PROCESSED_DIR / "hybrid_lgbm_model.pkl"

In [26]:
import pickle
with open(model_path, "wb") as f:
    pickle.dump(ranker, f)

In [27]:
print("Saved hybrid model to:", model_path)

Saved hybrid model to: /Users/sanjaydilip/Desktop/Code/Projects/movielens-split/movielens-recommender-system/data/processed/hybrid_lgbm_model.pkl


# Summary

This notebook trained the hybrid ranking model.

It used the features created earlier, including:

- user taste profiles

- movie genre vectors

- similarity features

- ALS scores

The model learned how to score movies for each user.

The output is a saved model file.